In [ ]:
# Copyright 2026 50Hertz Transmission GmbH and Elia Transmission Belgium
#
# This Source Code Form is subject to the terms of the Mozilla Public License, v. 2.0.
# If a copy of the MPL was not distributed with this file,
# you can obtain one at https://mozilla.org/MPL/2.0/.
# Mozilla Public License, version 2.0

see https://github.com/powsybl/pypowsybl-jupyter

In [ ]:
import copy
import pypowsybl
from pypowsybl._pypowsybl import LoadFlowComponentStatus
from pypowsybl_jupyter import network_explorer
import numpy as np
import matplotlib.pyplot as plt

# Load CGMES network
# cgmes_zip = "YOUR_FILE.zip"
# network = pypowsybl.network.load(cgmes_zip)

# load UCTE network
# ucte_file = "YOUR_FILE.uct"
# network = pypowsybl.network.load(ucte_file)

# Load Powsybl Test network
# see powsybl_grids.py
# https://powsybl.readthedocs.io/projects/pypowsybl/en/stable/reference/network.html
network = pypowsybl.network.create_ieee57()
network.per_unit = True

# load pandapower network
# see pandapower_grids.py
# https://pandapower.readthedocs.io/en/stable/networks.html
# pandapower_net = pandapower.networks.case33bw()
# network = load_pandapower_net_for_powsybl(pandapower_net)  # type: pypowsybl.network.Network


# consider converting three-winding transformers to two-winding transformers
pypowsybl.network.replace_3_windings_transformers_with_3_2_windings_transformers(network)
# set grid into p.u.



provider_param = {
        "newtonRaphsonConvEpsPerEq": "1e-6",
        "maxNewtonRaphsonIterations": "20",
        "svcVoltageMonitoring": "false",
        "newtonRaphsonStoppingCriteriaType": "UNIFORM_CRITERIA",
        "generatorReactivePowerRemoteControl": "false",
        "useActiveLimits": "true",
        "phaseShifterRegulationOn": "false",
    }

powsybl_loadflow_parameter = pypowsybl.loadflow.Parameters(
        voltage_init_mode=pypowsybl.loadflow.VoltageInitMode.PREVIOUS_VALUES,
        transformer_voltage_control_on=False,
        use_reactive_limits=True,
        phase_shifter_regulation_on=False,
        twt_split_shunt_admittance=True,
        shunt_compensator_voltage_control_on=False,
        read_slack_bus=True,
        write_slack_bus=None,
        distributed_slack=True,
        dc_use_transformer_ratio=True,
        countries_to_balance=None,  # Sequence[str]
        connected_component_mode=pypowsybl.loadflow.ConnectedComponentMode.MAIN,  # ConnectedComponentMode
        dc_power_factor=None,
        provider_parameters=provider_param,  # Dict[str, str]
)

provider_param_one_step = {
        "newtonRaphsonConvEpsPerEq": "1e30",
        "maxNewtonRaphsonIterations": "20",
        "svcVoltageMonitoring": "false",
        "newtonRaphsonStoppingCriteriaType": "UNIFORM_CRITERIA",
        "generatorReactivePowerRemoteControl": "false",
        "useActiveLimits": "false",
        "phaseShifterRegulationOn": "false",
    }

# run a load flow and use the parameter used for DC+
loadflow_res = pypowsybl.loadflow.run_ac(network, parameters=powsybl_loadflow_parameter)[0]
if loadflow_res.status != LoadFlowComponentStatus.CONVERGED:
    raise ValueError(f"Load flow did not converge: {loadflow_res.status}")
# or run DC load flow



# want to disconnect a branch?
# get the branch id from network.get_branches().index
network.disconnect('L25-30-1')

bus_res_n0 = network.get_buses(attributes=["v_mag"])

powsybl_loadflow_parameter_one_step:pypowsybl.loadflow.Parameters = copy.copy(powsybl_loadflow_parameter)
powsybl_loadflow_parameter_one_step.provider_parameters = provider_param_one_step
powsybl_loadflow_parameter_one_step.use_reactive_limits = False
powsybl_loadflow_parameter_one_step.distributed_slack = False
loadflow_res_one_step = pypowsybl.loadflow.run_ac(network, parameters=powsybl_loadflow_parameter_one_step)[0]
if loadflow_res_one_step.iteration_count != 1:
    raise ValueError(f"DC+ simulation failed")
print(f"Load flow status with one step parameters: {loadflow_res_one_step}")


bus_res_dcplus = network.get_buses(attributes=["v_mag"])
loadflow_res = pypowsybl.loadflow.run_ac(network, parameters=powsybl_loadflow_parameter)[0]
print(f"Load flow status with ac parameters: {loadflow_res}")
bus_res_ac = network.get_buses(attributes=["v_mag"])
res = bus_res_dcplus.merge(bus_res_ac, on="id", suffixes=("_dcplus", "_ac"))
res = res.merge(bus_res_n0, on="id")
res.sort_values("v_mag_ac", inplace=True)


# scatter plot of voltage magnitudes before and after load flow
fig_magnitude, ax_magnitude = plt.subplots(figsize=(6, 6))
mask = ~np.isnan(res["v_mag_ac"]) & ~np.isnan(res["v_mag_dcplus"])
x_vals = res.loc[mask, "v_mag_ac"].to_numpy()
dcplus_vals = res.loc[mask, "v_mag_dcplus"].to_numpy()
n0_vals = res.loc[mask, "v_mag"].to_numpy()

if x_vals.size == 0:
    raise ValueError("No finite voltage data available for plotting.")

combined_vals = np.concatenate((x_vals, dcplus_vals, n0_vals))
v_min = combined_vals.min()
v_max = combined_vals.max()

ax_magnitude.scatter(x_vals, dcplus_vals, s=12, alpha=0.6, label="DC+")
ax_magnitude.scatter(
    x_vals,
    n0_vals,
    s=36,
    marker="*",
    facecolors="none",
    edgecolors="C1",
    linewidths=1.2,
    label="N0 (original)",
)
ax_magnitude.plot([v_min, v_max], [v_min, v_max], color="k", linestyle="--", linewidth=1, label="AC = reference")

if x_vals.size > 1:
    corr = np.corrcoef(x_vals, dcplus_vals)[0, 1]
    ax_magnitude.text(0.02, 0.98, f"r={corr:.3f}", transform=ax_magnitude.transAxes, va="top", fontsize=12)

x_lim_offset = 0.005

ax_magnitude.set_xlim(v_min - x_lim_offset, v_max + x_lim_offset)
ax_magnitude.set_ylim(v_min - x_lim_offset, v_max + x_lim_offset)
ax_magnitude.set_xlabel("Voltage magnitude (AC, p.u.)")
ax_magnitude.set_ylabel("Voltage magnitude (DC+ / N0, p.u.)")
ax_magnitude.set_title("Voltage magnitude comparison")
ax_magnitude.legend(frameon=False)
ax_magnitude.grid(False)
plt.tight_layout()


In [ ]:
# # param sld
# component_library = "Convergence"
# # component_library = "FlatDesign"
# sld_param = pypowsybl.network.SldParameters(
#     use_name=True, component_library=component_library, nodes_infos=True, display_current_feeder_info=True
# )
# nad_parameters = pypowsybl.network.NadParameters(edge_info_along_edge=True, substation_description_displayed=True)

# # uses NadParameters and SldParameters to customize the diagrams
# network_explorer(
#     network, depth=2, sld_parameters=sld_param, nad_parameters=nad_parameters
# )